In [2]:

# DAY 3 - TASK 1
import os
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder

os.makedirs("../outputs", exist_ok=True)

print("FETCHING ADULT DATASET")
adult = fetch_openml(
    "adult",
    version=2,
    as_frame=True
)

df = adult.frame.copy()

print("Dataset loaded successfully!")
print("Original dataset shape:", df.shape)
df = df.replace("?", np.nan)
print("\n'?' values replaced with NaN.")
#  CREATE TARGET
df["target"] = df["class"].map({
    "<=50K": 0,
    ">50K": 1
}).astype(int)

print("\nTarget distribution:")
print(df["target"].value_counts())

#  CREATE COPY FOR FEATURE ENGINEERING
engineered_df = df.copy()
#  ENGINEERED FEATURE 1
# AGE BUCKET

engineered_df["age_bucket"] = pd.cut(
    engineered_df["age"],
    bins=[0, 25, 35, 45, 55, 65, np.inf],
    labels=[
        "Young",
        "Early_Career",
        "Mid_Career",
        "Experienced",
        "Senior",
        "Older"
    ],
    include_lowest=True
)
#  ENGINEERED FEATURE 2
# HOURS PER WEEK BUCKET
engineered_df["hours_bucket"] = pd.cut(
    engineered_df["hours-per-week"],
    bins=[0, 30, 40, 50, 60, np.inf],
    labels=[
        "Part_Time",
        "Standard",
        "Overtime",
        "High_Hours",
        "Very_High_Hours"
    ],
    include_lowest=True
)
#  ENGINEERED FEATURE 3
# CAPITAL GAIN FLAG

engineered_df["capital_gain_flag"] = (
    engineered_df["capital-gain"] > 0
).astype(int)


# ENGINEERED FEATURE 4
# LOG CAPITAL GAIN

engineered_df["log_capital_gain"] = np.log1p(
    engineered_df["capital-gain"]
)

# ENGINEERED FEATURE 5
# HIGHER EDUCATION BOOLEAN
# education-num >= 13 represents approximately
# Bachelor's degree or higher

engineered_df["higher_education"] = (
    engineered_df["education-num"] >= 13
).astype(int)

#ENGINEERED FEATURE 6
#EDUCATION & HOURS INTERACTION
engineered_df["education_hours_interaction"] = (
    engineered_df["education-num"]
    * engineered_df["hours-per-week"]
)
#ENGINEERED FEATURE 7
#CAPITAL LOSS FLAG
engineered_df["capital_loss_flag"] = (
    engineered_df["capital-loss"] > 0
).astype(int)

# ENGINEERED FEATURE 8
# AGE & HOURS INTERACTION
engineered_df["age_hours_interaction"] = (
    engineered_df["age"]
    * engineered_df["hours-per-week"]
)
#FEATURE DICTIONARY
feature_dictionary = pd.DataFrame({
    "Feature": [
        "age_bucket",
        "hours_bucket",
        "capital_gain_flag",
        "log_capital_gain",
        "higher_education",
        "education_hours_interaction",
        "capital_loss_flag",
        "age_hours_interaction"
    ],
    "Type": [
        "Categorical",
        "Categorical",
        "Binary",
        "Numeric",
        "Binary",
        "Numeric",
        "Binary",
        "Numeric"
    ],
    "Creation Rule": [
        "Age divided into meaningful age groups.",
        "Hours-per-week divided into work-hour groups.",
        "1 if capital-gain > 0, otherwise 0.",
        "log(1 + capital-gain).",
        "1 if education-num >= 13, otherwise 0.",
        "education-num multiplied by hours-per-week.",
        "1 if capital-loss > 0, otherwise 0.",
        "age multiplied by hours-per-week."
    ],
    "Justification": [
        "Different age groups may have different income levels.",
        "Working-hour categories may capture part-time and overtime effects.",
        "Having any capital gain may distinguish higher-income individuals.",
        "Reduces the effect of extreme capital-gain values and reduces skew.",
        "Higher education is likely associated with higher income.",
        "Combines education level and working effort into one feature.",
        "Capital loss information may help distinguish income groups.",
        "Combines age and working hours to capture career/work effects."
    ]
})
#DISPLAY ENGINEERED FEATURES
print("ENGINEERED FEATURE DICTIONARY")
print(
    feature_dictionary.to_string(index=False)
)
# PREPARE FEATURES FOR MUTUAL INFORMATION


engineered_features = [
    "age_bucket",
    "hours_bucket",
    "capital_gain_flag",
    "log_capital_gain",
    "higher_education",
    "education_hours_interaction",
    "capital_loss_flag",
    "age_hours_interaction"
]

# CREATE DATA FOR MUTUAL INFORMATION
mi_data = engineered_df[
    engineered_features
].copy()

target = engineered_df["target"]

# HANDLE MISSING VALUES
# Categorical features
categorical_engineered = [
    "age_bucket",
    "hours_bucket"
]
for column in categorical_engineered:

    mi_data[column] = (
        mi_data[column]
        .astype(object)
        .fillna("Missing")
    )

# Numeric features
numeric_engineered = [
    "capital_gain_flag",
    "log_capital_gain",
    "higher_education",
    "education_hours_interaction",
    "capital_loss_flag",
    "age_hours_interaction"
]
for column in numeric_engineered:
    mi_data[column] = (
        pd.to_numeric(
            mi_data[column],
            errors="coerce"
        )
        .fillna(0)
    )
# ENCODE CATEGORICAL FEATURES
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)
mi_data[categorical_engineered] = (
    encoder.fit_transform(
        mi_data[categorical_engineered]
    )
)
#CONVERT EVERYTHING TO NUMERIC
mi_data = mi_data.astype(float)

#  CALCULATE MUTUAL INFORMATION
# True for categorical engineered features

discrete_features = [
    True,   # age_bucket
    True,   # hours_bucket
    True,   # capital_gain_flag
    False,  # log_capital_gain
    True,   # higher_education
    False,  # education_hours_interaction
    True,   # capital_loss_flag
    False   # age_hours_interaction
]
print("CALCULATING MUTUAL INFORMATION SCORES")
mi_scores = mutual_info_classif(
    mi_data,
    target,
    discrete_features=discrete_features,
    random_state=42
)
#ADD MUTUAL INFORMATION TO FEATURE DICTIONARY
mi_score_df = pd.DataFrame({
    "Feature": engineered_features,
    "Mutual_Information": mi_scores
})
feature_dictionary = feature_dictionary.merge(
    mi_score_df,
    on="Feature"
)
# Sort by predictive signal
feature_dictionary = (
    feature_dictionary
    .sort_values(
        by="Mutual_Information",
        ascending=False
    )
    .reset_index(drop=True)
)
#DISPLAY FINAL FEATURE DICTIONARY
print("FINAL FEATURE DICTIONARY WITH PREDICTIVE SIGNAL")
print(
    feature_dictionary.to_string(index=False)
)
# CREATE SHORT SIGNAL INTERPRETATION
print("UNIVARIATE PREDICTIVE SIGNAL INTERPRETATION")
for _, row in feature_dictionary.iterrows():

    score = row["Mutual_Information"]

    if score >= 0.10:
        signal = "Strong univariate signal"

    elif score >= 0.03:
        signal = "Moderate univariate signal"

    elif score > 0:
        signal = "Weak univariate signal"

    else:
        signal = "Very weak/no univariate signal"

    print(
        f"{row['Feature']}: "
        f"MI = {score:.4f} -> {signal}"
    )
# SAVE FEATURE DICTIONARY
feature_dictionary.to_csv(
    "../outputs/day3_task1_feature_dictionary.csv",
    index=False
)
# SAVE ENGINEERED DATASET
engineered_df.to_csv(
    "../outputs/day3_engineered_adult_dataset.csv",
    index=False
)
# DISPLAY NEW FEATURE VALUES
print("\n" + "=" * 70)
print("SAMPLE OF ENGINEERED FEATURES")
print("=" * 70)
print(
    engineered_df[
        [
            "age",
            "hours-per-week",
            "capital-gain",
            "education-num",
            "age_bucket",
            "hours_bucket",
            "capital_gain_flag",
            "log_capital_gain",
            "higher_education",
            "education_hours_interaction",
            "capital_loss_flag",
            "age_hours_interaction",
            "target"
        ]
    ].head(10).to_string(index=False)
)



FETCHING ADULT DATASET
Dataset loaded successfully!
Original dataset shape: (48842, 15)

'?' values replaced with NaN.

Target distribution:
target
0    37155
1    11687
Name: count, dtype: int64
ENGINEERED FEATURE DICTIONARY
                    Feature        Type                                 Creation Rule                                                       Justification
                 age_bucket Categorical       Age divided into meaningful age groups.              Different age groups may have different income levels.
               hours_bucket Categorical Hours-per-week divided into work-hour groups. Working-hour categories may capture part-time and overtime effects.
          capital_gain_flag      Binary           1 if capital-gain > 0, otherwise 0.  Having any capital gain may distinguish higher-income individuals.
           log_capital_gain     Numeric                        log(1 + capital-gain). Reduces the effect of extreme capital-gain values and reduces skew.
    